In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

from sklearn.metrics.pairwise import cosine_similarity

### Loading Animelist

In [2]:
df_AnimeList = pd.DataFrame()
for f in list(Path('../Scraping/Season').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_AnimeList = pd.concat([df_AnimeList, df_Read])

del df_Read

In [3]:
df_AnimeList['MAL_id'] = df_AnimeList['Link'].apply(lambda x : int(re.findall(r'anime/(\d+)',x)[0]))
df_AnimeList = df_AnimeList.drop(columns=['Link', 'Unnamed: 0'])
df_AnimeList = df_AnimeList.set_index('MAL_id')
df_AnimeList = df_AnimeList.drop_duplicates()
df_AnimeList

,Title
MAL_id,
30,Shinseiki Evangelion
852,Gokinjo Monogatari
495,Bakuretsu Hunters
1567,Kaitou Saint Tail
116,Shinpi no Sekai El-Hazard (TV)
...,...
54145,Cardfight!! Vanguard: D Series Season 9
53722,Bullet/Bullet
60726,My Melody & Kuromi


### Loading User Ratings

In [4]:
df_Raw = pd.DataFrame()
for f in list(Path('../Scraping/Users/Watchlist').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_Read['User'] = f.name.replace('.csv', '')
    df_Raw = pd.concat([df_Raw, df_Read])

del df_Read

In [5]:
df_Raw = df_Raw.drop(columns=['Unnamed: 0'])
df_Raw = df_Raw.dropna()

df_Raw['MAL_id'] = df_Raw['MAL_id'].astype(int)
df_Raw

,MAL_id,Title,Score,Type,User
4,40028,Shingeki_no_Kyojin__The_Final_Season,10.0,TV,-Norza-
20,40938,Hige_wo_Soru_Soshite_Joshikousei_wo_Hirou,9.0,TV,-Norza-
32,1735,Naruto__Shippuuden,9.0,TV,-Norza-
59,3455,To_LOVE-Ru,7.0,TV,-Norza-
61,16011,Tokyo_Ravens,8.0,TV,-Norza-
...,...,...,...,...,...
223,36889,Oyasumi_Sex,7.0,OVA,_AverageWatcher_
224,49987,Papa_Katsu,6.0,OVA,_AverageWatcher_
226,59120,Shounen_ga_Otona_ni_Natta_Natsu,3.0,OVA,_AverageWatcher_
227,49329,Soshite_Watashi_wa_Sensei_ni,4.0,OVA,_AverageWatcher_


### User-Anime Score Matrix

In [6]:
matrix_Raw = df_Raw.pivot_table(index='User', columns='MAL_id', values='Score', aggfunc='mean')
matrix_Raw = matrix_Raw.fillna(0)
matrix_Raw = matrix_Raw.astype(np.int8)
matrix_Raw

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60108,60182,60407,60410,60443,60512,60544,60553,60629,60736
User,,,,,,,,,,,,,,,,,,,,,
-Norza-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0neiros,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24xnyk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2alvin,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ACRONYM117,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
xLucqd,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
xquasarx,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
yanzz22,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### User-User Similarity Matrix

In [7]:
matrix_Users = pd.DataFrame(cosine_similarity(matrix_Raw), index=matrix_Raw.index, columns=matrix_Raw.index)
matrix_Users.head(10)

User,-Norza-,0neiros,24xnyk,2alvin,ACRONYM117,Actual_Hana,Aelter,Ahegaofan94,AlphaSchnecke,Arcanon,...,themastergooner,traed,tytl,valoon,wellfuckthat,xLucqd,xquasarx,yanzz22,zairiko,zhorahh
User,,,,,,,,,,,,,,,,,,,,,
-Norza-,1.000000,0.162018,0.069213,0.00000,0.316455,0.096524,0.000000,0.000000,0.321150,0.207100,...,0.000000,0.039509,0.000000,0.000000,0.109467,0.144350,0.018825,0.000000,0.100901,0.000000
0neiros,0.162018,1.000000,0.072180,0.00000,0.218530,0.140174,0.008516,0.000000,0.142978,0.270482,...,0.000000,0.121981,0.000000,0.004780,0.203071,0.110429,0.072594,0.000000,0.156020,0.097975
24xnyk,0.069213,0.072180,1.000000,0.09202,0.044967,0.012534,0.073899,0.219096,0.047476,0.036936,...,0.055743,0.024401,0.183349,0.013401,0.000000,0.000000,0.000000,0.248038,0.015199,0.000000
2alvin,0.000000,0.000000,0.092020,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.120290,0.000000,0.000000,0.000000,0.000000,0.045529,0.000000,0.000000
ACRONYM117,0.316455,0.218530,0.044967,0.00000,1.000000,0.239364,0.000000,0.000000,0.258456,0.218094,...,0.000000,0.106998,0.000000,0.000000,0.143739,0.195954,0.148539,0.000000,0.193054,0.138530
Actual_Hana,0.096524,0.140174,0.012534,0.00000,0.239364,1.000000,0.000000,0.000000,0.167723,0.128019,...,0.000000,0.131124,0.000000,0.025087,0.050346,0.079326,0.157304,0.000000,0.242825,0.088072
Aelter,0.000000,0.008516,0.073899,0.00000,0.000000,0.000000,1.000000,0.164688,0.010821,0.009035,...,0.079863,0.000000,0.132643,0.000000,0.000000,0.000000,0.000000,0.088595,0.000000,0.000000
Ahegaofan94,0.000000,0.000000,0.219096,0.00000,0.000000,0.000000,0.164688,1.000000,0.008135,0.000000,...,0.000000,0.000000,0.328987,0.006549,0.000000,0.000000,0.000000,0.183229,0.000000,0.000000
AlphaSchnecke,0.321150,0.142978,0.047476,0.00000,0.258456,0.167723,0.010821,0.008135,1.000000,0.166593,...,0.088067,0.099379,0.000000,0.008677,0.072163,0.205767,0.056346,0.003203,0.155216,0.092196


## Test User Actual_Hana
Gather Top 10 Most Similar Users

In [8]:
USER = 'Actual_Hana'

similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:11]
similar_users

User
Deadpulse         0.288036
manakamukaido     0.269933
saxzer            0.259764
lYukki            0.257342
BELiCARiOs        0.256026
zairiko           0.242825
ACRONYM117        0.239364
NachoPizza        0.239337
Gabby_NEW         0.238397
SupaJanoskiiii    0.232487
Name: Actual_Hana, dtype: float64

## Shows User Watched. Others have Watched Some

In [9]:
alreadyWatched = matrix_Raw.loc[USER] > 0
matrix_Raw.loc[np.append(USER, similar_users.index), alreadyWatched]

MAL_id,20,22,121,123,226,270,355,356,384,853,...,22145,22199,22319,22535,23273,23283,23317,23673,27899,28223
User,,,,,,,,,,,,,,,,,,,,,
Actual_Hana,10,9,10,10,9,9,9,9,10,9,...,8,9,10,10,10,10,8,10,10,10
Deadpulse,0,0,0,0,0,0,0,0,0,0,...,0,0,10,0,8,0,0,0,6,0
manakamukaido,0,9,0,0,0,0,8,0,0,0,...,0,0,0,0,0,0,0,0,0,0
saxzer,9,0,0,9,0,0,0,10,0,0,...,9,0,7,0,7,0,9,7,5,10
lYukki,0,0,0,0,0,0,0,0,0,9,...,0,0,0,0,0,0,0,0,0,0
BELiCARiOs,0,0,0,0,0,0,0,0,0,9,...,0,0,0,8,10,0,0,9,0,8
zairiko,0,0,0,0,0,0,0,0,0,10,...,0,0,0,0,9,7,0,0,0,9
ACRONYM117,0,0,0,0,0,0,9,0,0,0,...,0,8,0,8,7,0,0,0,0,8
NachoPizza,0,0,0,0,0,0,9,0,0,0,...,0,8,10,9,10,0,0,0,8,0


## Shows User hasn't Watched. Others have

In [10]:
matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]
matrix_Other_User_Ratings

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60108,60182,60407,60410,60443,60512,60544,60553,60629,60736
User,,,,,,,,,,,,,,,,,,,,,
Deadpulse,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
manakamukaido,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
saxzer,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
lYukki,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
BELiCARiOs,0,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
zairiko,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ACRONYM117,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NachoPizza,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Gabby_NEW,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Recommendations' Avg Score by Similar Users

In [11]:
reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:10]
df_Reccomendations = df_AnimeList.loc[reccomended_Shows.index].copy()
df_Reccomendations['Avg_Score'] = reccomended_Shows.values
df_Reccomendations

,Title,Avg_Score
MAL_id,,
24833,Ansatsu Kyoushitsu,7.0
2904,Code Geass: Hangyaku no Lelouch R2,6.7
13759,Sakura-sou no Pet na Kanojo,6.6
9989,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,6.0
12189,Hyouka,5.9
14741,Chuunibyou demo Koi ga Shitai!,5.6
32281,Kimi no Na wa.,5.4
16067,Nagi no Asu kara,5.4
7054,Kaichou wa Maid-sama!,5.2


In [12]:
def GetPersonalTop(USER, n = 10):
    USER_top = matrix_Raw.loc[USER].sort_values(ascending=False)[:n]
    
    if USER_top.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[USER_top.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[USER_top.index].drop_duplicates()
    
    df['Personal_Score'] = USER_top.values
    df = df.rename(columns={'Title': f"{USER}'s Top {n}"})
    return df

    
def Recommend(USER, n = 10):
    similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:n+1]
    alreadyWatched = matrix_Raw.loc[USER] > 0
    matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]

    reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:n]

    if reccomended_Shows.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[reccomended_Shows.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[reccomended_Shows.index].drop_duplicates()
        
    df['Avg_Recommenders_Score'] = reccomended_Shows.values
    df = df.rename(columns={'Title': f"Others' Top {n}"})
        
    return df

## Test Mainstream Viewer

In [13]:
USER = '-Norza-'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,-Norza-'s Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,32380,Kono Subarashii Sekai ni Shukufuku wo!: Kono S...,10,37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,8.7
1,23847,Yahari Ore no Seishun Love Comedy wa Machigatt...,10,38329,Seishun Buta Yarou wa Yumemiru Shoujo no Yume ...,7.2
2,37105,Grand Blue,10,1535,Death Note,6.9
3,31704,One Punch Man: Road to Hero,10,23273,Shigatsu wa Kimi no Uso,6.7
4,42203,Re:Zero kara Hajimeru Isekai Seikatsu 2nd Seas...,10,9253,Steins;Gate,6.5
5,17265,Log Horizon,10,15809,Hataraku Maou-sama!,5.9
6,16498,Shingeki no Kyojin,10,9989,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,5.8
7,32281,Kimi no Na wa.,10,4224,Toradora!,5.7
8,39587,Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season,10,2904,Code Geass: Hangyaku no Lelouch R2,5.7
9,13759,Sakura-sou no Pet na Kanojo,10,4181,Clannad: After Story,5.7


## Test Watched Only One Punch Man (Cold Start)

In [14]:
USER = 'EXUPUROOOSHUN'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,EXUPUROOOSHUN's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,30276,One_Punch_Man,10,16498,Shingeki no Kyojin,6.1
1,36033,Kagirohi__Shaku_Kei_-_Another,0,32281,Kimi no Na wa.,5.8
2,36098,Kimi_no_Suizou_wo_Tabetai,0,31240,Re:Zero kara Hajimeru Isekai Seikatsu,5.6
3,36089,9,0,28851,Koe no Katachi,5.0
4,36064,Fate_Grand_Order,0,25777,Shingeki no Kyojin Season 2,4.6
5,36052,Kyonyuu_Daikazoku_Saimin,0,31964,Boku no Hero Academia,4.4
6,36051,Hitozuma_Life__One_Time_Gal,0,30831,Kono Subarashii Sekai ni Shukufuku wo!,4.3
7,36049,Dagashi_Kashi_2,0,23273,Shigatsu wa Kimi no Uso,3.9
8,36043,Net-juu_no_Susume_Special,0,21881,Sword Art Online II,3.8
9,36040,Saint_Seiya__Knights_of_the_Zodiac,0,37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,3.8


## AOT Fan

In [15]:
USER = 'Math082r'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,Math082r's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,2251,Baccano!,10,1575,Code Geass: Hangyaku no Lelouch,6.5
1,28851,Koe no Katachi,10,2904,Code Geass: Hangyaku no Lelouch R2,6.4
2,4181,Clannad: After Story,9,9989,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,5.5
3,3297,Aria the Origination,9,5114,Fullmetal Alchemist: Brotherhood,5.3
4,48583,Shingeki no Kyojin: The Final Season Part 2,9,14345,Btooom!,5.2
5,40028,Shingeki no Kyojin: The Final Season,9,11843,Danshi Koukousei no Nichijou,5.2
6,51535,Shingeki no Kyojin: The Final Season - Kankets...,9,9919,Ao no Exorcist,5.2
7,59571,Shingeki no Kyojin Movie: Kanketsu-hen - The L...,9,21,One Piece,5.2
8,39198,Kanata no Astra,8,28171,Shokugeki no Souma,5.0
9,14397,Chihayafuru 2,8,32282,Shokugeki no Souma: Ni no Sara,4.8


## Doraemon Addict + Porn Addict

In [16]:
USER = 'Muhclis'
pd.concat([GetPersonalTop(USER,100).reset_index(), Recommend(USER,100).reset_index()], axis=1)

,MAL_id,Muhclis's Top 100,Personal_Score,MAL_id,Others' Top 100,Avg_Recommenders_Score
0,39800,Joshi_Luck,10,24833,Ansatsu Kyoushitsu,3.37
1,2654,Doraemon_Movie_22__Nobita_to_Tsubasa_no_Yuusha...,10,30654,Ansatsu Kyoushitsu 2nd Season,3.02
2,2675,Doraemon_Movie_17__Nobita_to_Ginga_Express,10,31240,Re:Zero kara Hajimeru Isekai Seikatsu,3.00
3,2674,Doraemon_Movie_16__Nobita_no_Sousei_Nikki,10,28121,Dungeon ni Deai wo Motomeru no wa Machigatteir...,2.84
4,2673,Doraemon_Movie_27__Nobita_no_Shin_Makai_Daibou...,10,1575,Code Geass: Hangyaku no Lelouch,2.82
...,...,...,...,...,...,...
95,43793,Knight_of_Erin,10,35581,Oide yo! Mizuryuu Kei Land,1.26
96,35376,Himouto_Umaru-chan_R,10,14227,Tonari no Kaibutsu-kun,1.26
97,43969,Kanojo_mo_Kanojo,10,17074,Monogatari Series: Second Season,1.26
98,35363,Kobayashi-san_Chi_no_Maid_Dragon__Valentine_So...,10,11771,Kuroko no Basket,1.25


## Drama/Seinnen

In [18]:
USER = 'Shizawari'
pd.concat([GetPersonalTop(USER, 15).reset_index(), Recommend(USER, 15).reset_index()], axis=1)

,MAL_id,Shizawari's Top 15,Personal_Score,MAL_id,Others' Top 15,Avg_Recommenders_Score
0,33352,Violet_Evergarden,10,32281,Kimi no Na wa.,7.866667
1,56784,Bleach__Sennen_Kessen-hen_-_Soukoku-tan,10,38329,Seishun Buta Yarou wa Yumemiru Shoujo no Yume ...,7.133333
2,47917,Bocchi_the_Rock,10,30831,Kono Subarashii Sekai ni Shukufuku wo!,6.400000
3,9260,Kizumonogatari_I__Tekketsu-hen,10,1535,Death Note,5.733333
4,5081,Bakemonogatari,10,23273,Shigatsu wa Kimi no Uso,5.533333
5,52991,Sousou_no_Frieren,10,38691,Dr. Stone,5.466667
6,48653,Yoru_ni_Kakeru,10,45576,Mushoku Tensei: Isekai Ittara Honki Dasu Part 2,5.400000
7,50416,Skip_to_Loafer,10,42897,Horimiya,5.133333
8,55016,Idol,10,11061,Hunter x Hunter (2011),5.066667
9,37521,Vinland_Saga,10,14813,Yahari Ore no Seishun Love Comedy wa Machigatt...,5.066667


## Gooner

In [19]:
USER = 'omanko1'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,omanko1's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,49295,Chizuru-chan Kaihatsu Nikki,10,11827,Kuroinu: Kedakaki Seijo wa Hakudaku ni Somaru,7.3
1,59119,Imaria,10,38470,Saimin Seishidou,6.7
2,50192,Akane wa Tsumare Somerareru,10,24641,Baku Ane: Otouto Shibocchau zo! The Animation,6.3
3,39530,Ecchi na Oneechan ni Shiboraretai,9,36720,Youkoso! Sukebe Elf no Mori e,6.3
4,32587,Shoujo Ramune,9,39042,Gaki ni Modotte Yarinaoshi!!!,6.0
5,55146,Kunoichi Ryoujokuden Ajisai,9,48698,Mako-chan Kaihatsu Nikki,5.6
6,23479,Itadaki! Seieki♥ Special,9,33322,JK Bitch ni Shiboraretai,5.5
7,43466,Himawari wa Yoru ni Saku,9,38866,Seikatsu Shuukan The Animation,5.4
8,48468,Uchi no Otouto Maji de Dekain Dakedo Mi ni Konai?,9,29575,Mankitsu Happening,5.4
9,2539,Itazura The Animation,9,21829,Fella Pure: Mitarashi-san Chi no Jijou The Ani...,5.4
